In [10]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
import os
load_dotenv()

True

In [5]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

In [6]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [7]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = model.invoke(messages)

    return {'messages':[response]}

In [11]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [9]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['messages'][-1].content

'The capital of India is **New Delhi**.'

In [12]:
thread_id = '1'
while True:
    user_msg = input("Type here: ")
    print('You: ', user_msg)

    if(user_msg.strip().lower() in ['exit', 'bye', 'quit', 'stop']):
        break
    
    config = {'configurable':{'thread_id': thread_id}};
    response = chatbot.invoke({'messages':[HumanMessage(content = user_msg)]}, config=config)

    print('AI: ', response['messages'][-1].content)


You:  Hi, my name is Amrit
AI:  Hello Amrit! It's nice to meet you.

How can I assist you today?
You:  what's my name
AI:  Your name is Amrit.
You:  Yeah add 22 + 13 and give result
AI:  Okay, Amrit.

22 + 13 = **35**
You:  multiply 2 to result
AI:  Okay, Amrit.

The previous result was 35.

35 * 2 = **70**
You:  return the answer
AI:  The answer is **70**.
You:  exit
